# 🏎️ High-Accuracy, GPU-Optimized Stanford Cars ResNet50 Training
### Intelligent Fine-Grained Vehicle Recognition System

This notebook executes end-to-end training and evaluation for a torchvision **ResNet50** fine-grained vehicle classifier on the **Stanford Cars (196 classes)** dataset.

#### Key Highlights:
- **Cloud GPU Auto-Adaptation**: Detects NVIDIA L4, T4, A100, or V100; configures optimal batch size, workers, and mixed precision (`bf16` or `fp16`).
- **2-Stage Transfer Learning**:
  - **Stage A (Warmup)**: Freeze backbone, train 196-class classification head for 2 epochs.
  - **Stage B (Fine-Tuning)**: Unfreeze backbone, train end-to-end with discriminative learning rates (`1e-4` for backbone, `5e-4` for head) using AdamW, cosine annealing, and label smoothing.
- **Channels-Last Memory Layout**: Maximizes Tensor Core throughput.
- **Persistent Google Drive Resumption**: Checkpoints are auto-saved to Google Drive every epoch; if Colab disconnects, training resumes from the last completed epoch.
- **Multi-Attribute Evaluation**: Evaluates Top-1, Top-5, Make accuracy, Model accuracy, Year accuracy, inference latency, and saves training curves.


## 1. Cloud GPU Inspection & Environment Setup


In [ ]:
import os
import sys
import time
import json
import shutil
import random
from pathlib import Path

import torch
import torch.nn as nn
import torchvision
import torchvision.models as tv_models
import torchvision.transforms as T
import numpy as np
import pandas as pd
from PIL import Image

# 1. Hardware Detection
print("=" * 60)
print("HARDWARE & PYTORCH ENVIRONMENT")
print("=" * 60)
print(f"PyTorch Version: {torch.__version__}")
print(f"Torchvision Version: {torchvision.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    cc = torch.cuda.get_device_capability(0)
    print(f"GPU Device: {gpu_name}")
    print(f"VRAM: {vram_gb:.2f} GB")
    print(f"Compute Capability: {cc[0]}.{cc[1]}")

    # Optimal Configuration
    if vram_gb >= 20.0:  # A100 / L4
        BATCH_SIZE = 64
        PRECISION = "bf16" if cc[0] >= 8 else "fp16"
        AMP_DTYPE = torch.bfloat16 if cc[0] >= 8 else torch.float16
        NUM_WORKERS = 4
    elif vram_gb >= 12.0:  # T4 / V100
        BATCH_SIZE = 32
        PRECISION = "fp16"
        AMP_DTYPE = torch.float16
        NUM_WORKERS = 2
    else:
        BATCH_SIZE = 16
        PRECISION = "fp16"
        AMP_DTYPE = torch.float16
        NUM_WORKERS = 2

    CHANNELS_LAST = True
    torch.backends.cudnn.benchmark = True
    print(f"-> Selected Batch Size: {BATCH_SIZE}")
    print(f"-> Mixed Precision: {PRECISION}")
    print(f"-> DataLoader Workers: {NUM_WORKERS}")
else:
    device = torch.device("cpu")
    BATCH_SIZE = 16
    PRECISION = "fp32"
    AMP_DTYPE = torch.float32
    NUM_WORKERS = 0
    CHANNELS_LAST = False
    print("WARNING: Running on CPU. GPU runtime strongly recommended!")
print("=" * 60)


## 2. Google Drive Mounting & Persistent Checkpoint Setup


In [ ]:
# Mount Google Drive for persistent checkpoint storage
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_CHECKPOINT_DIR = Path('/content/drive/MyDrive/stanford_cars_checkpoints')
    DRIVE_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted. Checkpoint directory: {DRIVE_CHECKPOINT_DIR}")
except Exception as e:
    print(f"Google Drive not mounted ({e}). Checkpoints will be saved locally to ./outputs/classification/checkpoints")
    DRIVE_CHECKPOINT_DIR = None

LOCAL_OUTPUT_DIR = Path('/content/outputs/classification')
LOCAL_CKPT_DIR = LOCAL_OUTPUT_DIR / 'checkpoints'
LOCAL_METRICS_DIR = LOCAL_OUTPUT_DIR / 'metrics'
LOCAL_PLOTS_DIR = LOCAL_OUTPUT_DIR / 'plots'

for p in (LOCAL_CKPT_DIR, LOCAL_METRICS_DIR, LOCAL_PLOTS_DIR):
    p.mkdir(parents=True, exist_ok=True)


## 3. Stanford Cars Dataset Ingestion
Downloads the **Stanford Cars (196 classes)** dataset from Kaggle (`eduardo4jesus/stanford-cars-dataset`).
You can provide your Kaggle credentials via Colab Secrets (`userdata`), environment variables, or upload your `kaggle.json`.


In [ ]:
import getpass

DATASET_DIR = Path('/content/dataset')
DATASET_DIR.mkdir(parents=True, exist_ok=True)

# Setup Kaggle Credentials
kaggle_json_path = Path.home() / '.kaggle' / 'kaggle.json'
if not kaggle_json_path.is_file():
    kaggle_json_path.parent.mkdir(parents=True, exist_ok=True)

    # Try Colab Secrets first
    try:
        from google.colab import userdata
        k_user = userdata.get('KAGGLE_USERNAME')
        k_key = userdata.get('KAGGLE_KEY')
        if k_user and k_key:
            with open(kaggle_json_path, 'w') as f:
                json.dump({"username": k_user, "key": k_key}, f)
            print("Loaded Kaggle credentials from Colab Secrets.")
    except Exception:
        pass

if not kaggle_json_path.is_file():
    if 'KAGGLE_USERNAME' in os.environ and 'KAGGLE_KEY' in os.environ:
        with open(kaggle_json_path, 'w') as f:
            json.dump({"username": os.environ['KAGGLE_USERNAME'], "key": os.environ['KAGGLE_KEY']}, f)
    else:
        print("Please enter your Kaggle credentials to download Stanford Cars:")
        username = input("Kaggle Username: ").strip()
        key = getpass.getpass("Kaggle API Key: ").strip()
        with open(kaggle_json_path, 'w') as f:
            json.dump({"username": username, "key": key}, f)

os.chmod(kaggle_json_path, 0o600)

# Download and extract dataset if not already present
train_folder = DATASET_DIR / 'train'
test_folder = DATASET_DIR / 'test'

if not train_folder.is_dir() or len(list(train_folder.glob('*'))) < 190:
    print("Downloading eduardo4jesus/stanford-cars-dataset from Kaggle...")
    !pip install -q kaggle
    !kaggle datasets download -d eduardo4jesus/stanford-cars-dataset -p /content/dataset --unzip
    print("Dataset downloaded and extracted!")

# Handle potential nested folder structures
if (DATASET_DIR / 'Stanford Dataset' / 'train').is_dir():
    train_folder = DATASET_DIR / 'Stanford Dataset' / 'train'
    test_folder = DATASET_DIR / 'Stanford Dataset' / 'test'

n_train_classes = len([d for d in train_folder.iterdir() if d.is_dir()])
n_train_imgs = len(list(train_folder.glob('*/*.jpg')))
print(f"Verified dataset: {n_train_classes} classes, {n_train_imgs} training images.")


## 4. Taxonomy Alignment & High-Throughput DataLoaders


In [ ]:
# Build canonical 196-class taxonomy mapping
# Note: handles filesystem edge case 'Ram C_V Cargo Van Minivan 2012' <-> 'Ram C/V Cargo Van Minivan 2012'
class_dirs = sorted([d.name for d in train_folder.iterdir() if d.is_dir()])
canonical_taxonomy = {}
name_to_id = {}

for cid, dirname in enumerate(class_dirs):
    clean_name = dirname.replace("C_V", "C/V")
    parts = clean_name.split()
    make = parts[0] if parts else ""
    try:
        year = int(parts[-1])
        model = " ".join(parts[1:-1])
    except ValueError:
        year = -1
        model = " ".join(parts[1:])

    canonical_taxonomy[cid] = {
        "class_id": cid,
        "class_name": clean_name,
        "make": make,
        "model": model,
        "year": year
    }
    name_to_id[clean_name] = cid
    name_to_id[dirname] = cid

print(f"Indexed {len(canonical_taxonomy)} vehicle classes. Sample:")
print(f"  Class 0: {canonical_taxonomy[0]['class_name']} (Make: {canonical_taxonomy[0]['make']}, Year: {canonical_taxonomy[0]['year']})")

# Collect Samples
train_samples = []
for c_dir in sorted(train_folder.iterdir()):
    if not c_dir.is_dir():
        continue
    cid = name_to_id[c_dir.name]
    for img_p in sorted(c_dir.glob("*.jpg")):
        train_samples.append({"image_path": str(img_p), "class_id": cid, "class_name": canonical_taxonomy[cid]["class_name"]})

val_samples = []
if test_folder.is_dir():
    for c_dir in sorted(test_folder.iterdir()):
        if not c_dir.is_dir():
            continue
        cid = name_to_id.get(c_dir.name, name_to_id.get(c_dir.name.replace("C_V", "C/V"), -1))
        if cid != -1:
            for img_p in sorted(c_dir.glob("*.jpg")):
                val_samples.append({"image_path": str(img_p), "class_id": cid, "class_name": canonical_taxonomy[cid]["class_name"]})

if len(val_samples) < 1000:
    # Perform stratified split if test folder is not available
    rng = random.Random(42)
    by_class = {}
    for s in train_samples:
        by_class.setdefault(s["class_id"], []).append(s)
    train_samples, val_samples = [], []
    for cid, items in by_class.items():
        rng.shuffle(items)
        n_val = max(1, int(len(items) * 0.15))
        val_samples.extend(items[:n_val])
        train_samples.extend(items[n_val:])

print(f"Data Splits: {len(train_samples)} training samples, {len(val_samples)} validation samples.")

# Dataset Class
class FastStanfordCarsDataset(torch.utils.data.Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        rec = self.samples[idx]
        with Image.open(rec["image_path"]) as raw_img:
            img = raw_img.convert("RGB")
        tensor = self.transform(img) if self.transform else T.functional.to_tensor(img)
        return {"image": tensor, "class_id": torch.tensor(rec["class_id"], dtype=torch.long)}

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.8, 1.0)),
    T.RandomHorizontalFlip(p=0.5),
    T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.05),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_tf = T.Compose([
    T.Resize((256, 256)),
    T.CenterCrop((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_loader = DataLoader(
    FastStanfordCarsDataset(train_samples, transform=train_tf),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

val_loader = DataLoader(
    FastStanfordCarsDataset(val_samples, transform=val_tf),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available()
)
print("DataLoaders initialized successfully.")


## 5. ResNet50 Architecture & 2-Stage Training Configuration


In [ ]:
# Instantiate ResNet50 with ImageNet pretrained weights
print("Loading torchvision ResNet50 (ImageNet Pretrained)...")
weights = tv_models.ResNet50_Weights.DEFAULT
model = tv_models.resnet50(weights=weights)

# Replace final FC head with 196-class Linear layer + Dropout(0.2)
in_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(in_features, 196)
)

if CHANNELS_LAST:
    model = model.to(memory_format=torch.channels_last)
model = model.to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.05).to(device)
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

# Helper functions for Stage A (frozen) vs Stage B (fine-tuning)
def freeze_backbone(m):
    for name, p in m.named_parameters():
        p.requires_grad = ("fc" in name)
    print("Stage A: Backbone frozen. Only classification head trainable.")

def unfreeze_backbone(m):
    for p in m.parameters():
        p.requires_grad = True
    print("Stage B: Backbone unfrozen for discriminative fine-tuning.")

TOTAL_EPOCHS = 30
WARMUP_EPOCHS = 2
EARLY_STOPPING_PATIENCE = 5
BACKBONE_LR = 1e-4
HEAD_LR = 5e-4
WEIGHT_DECAY = 1e-4

# Check if resumable checkpoint exists in Drive
start_epoch = 1
best_val_top1 = 0.0
best_epoch = 0
history = []

resume_ckpt_path = None
if DRIVE_CHECKPOINT_DIR and (DRIVE_CHECKPOINT_DIR / 'last_checkpoint.pt').is_file():
    resume_ckpt_path = DRIVE_CHECKPOINT_DIR / 'last_checkpoint.pt'
elif (LOCAL_CKPT_DIR / 'last_checkpoint.pt').is_file():
    resume_ckpt_path = LOCAL_CKPT_DIR / 'last_checkpoint.pt'

if resume_ckpt_path:
    print(f"Resuming training from checkpoint: {resume_ckpt_path}")
    checkpoint = torch.load(resume_ckpt_path, map_location=device)
    model.load_state_dict(checkpoint["state_dict"])
    start_epoch = checkpoint.get("epoch", 0) + 1
    best_val_top1 = checkpoint.get("best_val_acc", 0.0)
    history = checkpoint.get("history", [])
    if "scaler_state_dict" in checkpoint and checkpoint["scaler_state_dict"]:
        scaler.load_state_dict(checkpoint["scaler_state_dict"])
    print(f"Resumed from epoch {start_epoch - 1} with previous Best Val Top-1: {best_val_top1 * 100.0:.2f}%")


## 6. Training Execution with Live Metrics & Persistent Drive Backup


In [ ]:
import matplotlib.pyplot as plt

patience_counter = 0

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    is_stage_a = (epoch <= WARMUP_EPOCHS)

    # Configure Optimizer according to current stage
    if is_stage_a:
        if epoch == start_epoch or epoch == 1:
            freeze_backbone(model)
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=HEAD_LR, weight_decay=WEIGHT_DECAY
            )
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=WARMUP_EPOCHS, eta_min=1e-5)
    else:
        if epoch == (WARMUP_EPOCHS + 1) or (epoch == start_epoch and start_epoch > WARMUP_EPOCHS):
            unfreeze_backbone(model)
            backbone_p = [p for n, p in model.named_parameters() if "fc" not in n and p.requires_grad]
            head_p = [p for n, p in model.named_parameters() if "fc" in n and p.requires_grad]
            optimizer = torch.optim.AdamW([
                {"params": backbone_p, "lr": BACKBONE_LR},
                {"params": head_p, "lr": HEAD_LR}
            ], weight_decay=WEIGHT_DECAY)
            remaining_epochs = max(1, TOTAL_EPOCHS - WARMUP_EPOCHS)
            scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=remaining_epochs, eta_min=1e-6)

    # 1. Training Epoch
    model.train()
    running_loss, correct_top1, correct_top5, total_s = 0.0, 0, 0, 0
    t0 = time.time()

    for batch_idx, batch in enumerate(train_loader):
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["class_id"].to(device, non_blocking=True)
        if CHANNELS_LAST:
            images = images.to(memory_format=torch.channels_last)

        optimizer.zero_grad()
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available(), dtype=AMP_DTYPE):
            logits = model(images)
            loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        bs = targets.size(0)
        running_loss += loss.item() * bs
        total_s += bs

        with torch.no_grad():
            top1 = logits.argmax(dim=1)
            correct_top1 += (top1 == targets).sum().item()
            k = min(5, logits.size(1))
            top5 = logits.topk(k, dim=1)[1]
            correct_top5 += top5.eq(targets.view(-1, 1).expand_as(top5)).sum().item()

    train_loss = running_loss / max(1, total_s)
    train_top1 = correct_top1 / max(1, total_s)

    # 2. Validation Epoch
    model.eval()
    val_running_loss, val_correct_top1, val_correct_top5, val_total_s = 0.0, 0, 0, 0
    with torch.no_grad():
        for batch in val_loader:
            images = batch["image"].to(device, non_blocking=True)
            targets = batch["class_id"].to(device, non_blocking=True)
            if CHANNELS_LAST:
                images = images.to(memory_format=torch.channels_last)

            logits = model(images)
            loss = criterion(logits, targets)

            bs = targets.size(0)
            val_running_loss += loss.item() * bs
            val_total_s += bs

            top1 = logits.argmax(dim=1)
            val_correct_top1 += (top1 == targets).sum().item()
            k = min(5, logits.size(1))
            top5 = logits.topk(k, dim=1)[1]
            val_correct_top5 += top5.eq(targets.view(-1, 1).expand_as(top5)).sum().item()

    val_loss = val_running_loss / max(1, val_total_s)
    val_top1 = val_correct_top1 / max(1, val_total_s)
    val_top5 = val_correct_top5 / max(1, val_total_s)
    scheduler.step()
    epoch_dur = time.time() - t0

    cur_lr = optimizer.param_groups[0]["lr"]
    stage_str = "Stage A (Warmup)" if is_stage_a else "Stage B (Fine-Tune)"
    print(f"Epoch {epoch:02d}/{TOTAL_EPOCHS:02d} ({stage_str}) [{epoch_dur:.1f}s] | "
          f"Train Loss: {train_loss:.4f} | Train Top-1: {train_top1*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Top-1: {val_top1*100:.2f}% | Val Top-5: {val_top5*100:.2f}% | LR: {cur_lr:.2e}")

    rec = {
        "epoch": epoch, "stage": stage_str, "train_loss": round(train_loss, 4),
        "train_top1": round(train_top1, 4), "val_loss": round(val_loss, 4),
        "val_top1": round(val_top1, 4), "val_top5": round(val_top5, 4),
        "learning_rate": cur_lr, "epoch_duration_s": round(epoch_dur, 1)
    }
    history.append(rec)

    # Checkpoint Persistence
    is_best = val_top1 > best_val_top1
    if is_best:
        best_val_top1 = val_top1
        best_epoch = epoch
        patience_counter = 0

    state = {
        "epoch": epoch, "architecture": "resnet50", "num_classes": 196,
        "state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "val_acc": val_top1, "best_val_acc": best_val_top1,
        "history": history, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")
    }

    # Save last
    torch.save(state, LOCAL_CKPT_DIR / "last_checkpoint.pt")
    if DRIVE_CHECKPOINT_DIR:
        shutil.copy2(LOCAL_CKPT_DIR / "last_checkpoint.pt", DRIVE_CHECKPOINT_DIR / "last_checkpoint.pt")

    # Save best
    if is_best:
        torch.save(state, LOCAL_CKPT_DIR / "best_checkpoint.pt")
        if DRIVE_CHECKPOINT_DIR:
            shutil.copy2(LOCAL_CKPT_DIR / "best_checkpoint.pt", DRIVE_CHECKPOINT_DIR / "best_checkpoint.pt")
        print(f"⭐️ New BEST validation Top-1 accuracy: {val_top1 * 100.0:.2f}% (Saved to Google Drive)")
    else:
        patience_counter += 1

    # Early stopping check
    if patience_counter >= EARLY_STOPPING_PATIENCE and epoch > WARMUP_EPOCHS:
        print(f"Early stopping triggered at epoch {epoch}. Best epoch was {best_epoch} ({best_val_top1*100:.2f}%).")
        break

print("Training finished!")


## 7. Multi-Attribute Evaluation & Visualizations


In [ ]:
# Load best checkpoint for evaluation
best_ckpt_file = LOCAL_CKPT_DIR / "best_checkpoint.pt"
if best_ckpt_file.is_file():
    ckpt = torch.load(best_ckpt_file, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    print(f"Loaded best checkpoint from epoch {ckpt['epoch']} (Val Top-1: {ckpt['best_val_acc']*100:.2f}%)")

model.eval()
all_targets, all_preds, all_top5, latencies = [], [], [], []

with torch.no_grad():
    for batch in val_loader:
        images = batch["image"].to(device, non_blocking=True)
        targets = batch["class_id"].tolist()
        if CHANNELS_LAST:
            images = images.to(memory_format=torch.channels_last)

        t0 = time.perf_counter()
        logits = model(images)
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append((time.perf_counter() - t0) * 1000.0 / len(targets))

        preds = logits.argmax(dim=1).cpu().tolist()
        top5 = logits.topk(5, dim=1)[1].cpu().tolist()
        all_targets.extend(targets)
        all_preds.extend(preds)
        all_top5.extend(top5)

n = len(all_targets)
top1_acc = sum(1 for p, t in zip(all_preds, all_targets) if p == t) / n
top5_acc = sum(1 for t, k_list in zip(all_targets, all_top5) if t in k_list) / n

# Multi-Attribute Accuracy (Make, Model, Year)
make_correct = sum(1 for p, t in zip(all_preds, all_targets) if canonical_taxonomy[p]['make'] == canonical_taxonomy[t]['make'])
model_correct = sum(1 for p, t in zip(all_preds, all_targets) if canonical_taxonomy[p]['model'] == canonical_taxonomy[t]['model'])
year_correct = sum(1 for p, t in zip(all_preds, all_targets) if canonical_taxonomy[p]['year'] == canonical_taxonomy[t]['year'] and canonical_taxonomy[t]['year'] != -1)

make_acc = make_correct / n
model_acc = model_correct / n
year_acc = year_correct / n
avg_latency = float(np.mean(latencies))
ckpt_size_mb = best_ckpt_file.stat().st_size / (1024**2)

eval_summary = {
    "total_samples": n,
    "top1_accuracy": round(top1_acc, 4),
    "top5_accuracy": round(top5_acc, 4),
    "attribute_metrics": {
        "make_accuracy": round(make_acc, 4),
        "model_accuracy": round(model_acc, 4),
        "year_accuracy": round(year_acc, 4),
        "derivation_note": "Decoded from 196-class predictions using canonical taxonomy"
    },
    "inference_latency_ms": round(avg_latency, 2),
    "checkpoint_size_mb": round(ckpt_size_mb, 2)
}

print("=" * 60)
print("FINAL EVALUATION METRICS:")
print("=" * 60)
print(f"Top-1 Accuracy:  {top1_acc*100:.2f}%")
print(f"Top-5 Accuracy:  {top5_acc*100:.2f}%")
print(f"Make Accuracy:   {make_acc*100:.2f}%")
print(f"Model Accuracy:  {model_acc*100:.2f}%")
print(f"Year Accuracy:   {year_acc*100:.2f}%")
print(f"Latency/Image:   {avg_latency:.2f} ms")
print(f"Model Size:      {ckpt_size_mb:.2f} MB")
print("=" * 60)

# Save Evaluation JSON & Class Mapping
with open(LOCAL_METRICS_DIR / "eval_results.json", "w") as f:
    json.dump(eval_summary, f, indent=2)
with open(LOCAL_METRICS_DIR / "class_mapping.json", "w") as f:
    json.dump({"num_classes": 196, "by_class_id": canonical_taxonomy}, f, indent=2)
with open(LOCAL_METRICS_DIR / "training_history.json", "w") as f:
    json.dump(history, f, indent=2)

# Generate and Save Training Curves
epochs = [h["epoch"] for h in history]
plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs, [h["train_loss"] for h in history], 'b-o', label='Train Loss')
plt.plot(epochs, [h["val_loss"] for h in history], 'r-s', label='Val Loss')
plt.title('Loss Curves')
plt.xlabel('Epoch')
plt.ylabel('Cross-Entropy Loss')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, [h["train_top1"]*100 for h in history], 'b-o', label='Train Top-1 (%)')
plt.plot(epochs, [h["val_top1"]*100 for h in history], 'g-s', label='Val Top-1 (%)')
plt.plot(epochs, [h["val_top5"]*100 for h in history], 'm-^', label='Val Top-5 (%)')
plt.title('Accuracy Curves')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()

plt.tight_layout()
plt.savefig(LOCAL_PLOTS_DIR / "training_history.png", dpi=150)
plt.show()
print("Saved plots and evaluation metrics.")


## 8. Export & Download Trained Checkpoint Package


In [ ]:
# Create zip package matching the repository structure
archive_path = '/content/stanford_cars_resnet50_artifacts'
shutil.make_archive(archive_path, 'zip', '/content/outputs')
zip_file = archive_path + '.zip'
print(f"Archive created: {zip_file} ({os.path.getsize(zip_file) / (1024**2):.2f} MB)")

# Also copy the zip to Google Drive if connected
if DRIVE_CHECKPOINT_DIR:
    shutil.copy2(zip_file, DRIVE_CHECKPOINT_DIR / 'stanford_cars_resnet50_artifacts.zip')
    print(f"Saved artifacts ZIP archive to Google Drive: {DRIVE_CHECKPOINT_DIR / 'stanford_cars_resnet50_artifacts.zip'}")

# Trigger browser download if running interactively in Colab
try:
    from google.colab import files
    print("Initiating browser download of trained model package...")
    files.download(zip_file)
except Exception:
    print(f"Download manually from file browser or Google Drive at: {zip_file}")
